In [46]:
import pandas as pd
import openpyxl
import numpy as np
import os
import re
from collections import Counter

In [129]:
pd.set_option('display.max_rows', 100) 
pd.set_option('display.max_columns', None) 

In [22]:
path = '/Users/emma/projects/cognoscolab/math_anxiety_LLMs'

In [23]:
data_2023_humans = pd.read_excel(f'{path}/data/raw_data_2023.xlsx', header=None)

## Association and Valence extraction

In [24]:
# get in the excel the chunk of data where the associations and valences are
valence_associations_raw = data_2023_humans.iloc[30:, 80:651]
# get the excel portion where the cue words can be found (first row 1:2, in the cols from 80 to 652, skipping every 12 cols)
cue_words_raw = data_2023_humans.iloc[1:2, 80:652:11]

# in the first cells of the first rows search for the cue word
cue_words = []
for col in cue_words_raw.columns:
    for cell in cue_words_raw[col]:
        # get te word between - - that can also have / (like studente/ssa)
        match = re.search(r'-\s*(\w+(?:/\w+)?)\s*-', str(cell))

        if match:
            cue_words.append(match.group(1))
        else:
            print(f"no match in col: {col+1}")

print(cue_words)
print('n of cue words:', len(cue_words))
print('n of cue words without duplicates:', len(set(cue_words)))
# 'rimandare' was presented twice, so the total of cue_words is 52, but with this duplicate

['tecnologia', 'matematica', 'ingegneria', 'biologia', 'fisica', 'statistica', 'scadenza', 'classe', 'esame', 'successo', 'intelligenza', 'studente/ssa', 'insegnante', 'genitore', 'compito', 'numeri', 'calcolo', 'curiosità', 'cambio_di_piani', 'tenere_traccia', 'studiare', 'note', 'pianificare', 'rimandare', 'fissare_un_obiettivo', 'memorizzare', 'programma', 'mappa_concettuale', 'revisionare', 'rimandare', 'impegno', 'motivazione', 'concentrazione', 'sfida', 'importanza', 'distrazione', 'sforzo', 'confusione', 'problema', 'errori', 'fallimento', 'utilità_della_matematica', 'test_di_matematica', 'ansia', 'lavagna', 'soluzione', 'performance_in_matematica', 'dimostrazione', 'matematico/a', 'algoritmo', 'programmare', 'equazione']
n of cue words: 52
n of cue words without duplicates: 51


### Valences

In [25]:
# get the cols containing the scores and the associated words
scores_cols = []
words_cols = []
for col in valence_associations_raw.columns:
    # check if, excluding the nan values, there are only number in that column
    if valence_associations_raw[col].dropna().apply(lambda x: str(x).isdigit()).all():
        # if so it's a column contaning scores (arousal or valence)
        scores_cols.append(col)
    else:
        # if not, it's a col containing the associated words
        words_cols.append(col)

In [26]:
# get all the scores data
all_scores = []
for col in scores_cols:
    all_scores.append(valence_associations_raw[col])
# concatenate all the scores values
all_scores_df = pd.concat(all_scores, axis=1)
# reassign the correct index to the df 
all_scores_df.columns = range(all_scores_df.shape[1])

# get only the columns containg the valence (the first 4 cols, dropping the second four)
cols_valence = []
for i in range(0, all_scores_df.shape[1], 8):
    cols_valence.extend(range(i, min(i + 4, all_scores_df.shape[1])))

# screate a df with all the data concerning the valence columns
valence_scores = all_scores_df.iloc[:, cols_valence].reset_index(drop=True)
# assign the indeces to the df 
valence_scores.columns = [f"val_{i}" for i in range(valence_scores.shape[1])]

### Words and participant_ids

In [27]:
# get the data for all the associated words
all_words = []
for col in words_cols:
    all_words.append(valence_associations_raw[col])
# concatenate the data of all the words
association_words = pd.concat(all_words, axis=1)
# set the indices of cols
association_words.columns = range(association_words.shape[1])
# reset index to match other dfs indices to later concatenate them
association_words = association_words.reset_index(drop=True)

In [28]:
# create participant_ids from the legth of the df
participant_ids = [f"student_exp2_{i+1:03}" for i in range(len(association_words))]
participant_ids

['student_exp2_001',
 'student_exp2_002',
 'student_exp2_003',
 'student_exp2_004',
 'student_exp2_005',
 'student_exp2_006',
 'student_exp2_007',
 'student_exp2_008',
 'student_exp2_009',
 'student_exp2_010',
 'student_exp2_011',
 'student_exp2_012',
 'student_exp2_013',
 'student_exp2_014',
 'student_exp2_015',
 'student_exp2_016',
 'student_exp2_017',
 'student_exp2_018',
 'student_exp2_019',
 'student_exp2_020',
 'student_exp2_021',
 'student_exp2_022',
 'student_exp2_023',
 'student_exp2_024',
 'student_exp2_025',
 'student_exp2_026',
 'student_exp2_027',
 'student_exp2_028',
 'student_exp2_029',
 'student_exp2_030',
 'student_exp2_031',
 'student_exp2_032',
 'student_exp2_033',
 'student_exp2_034',
 'student_exp2_035',
 'student_exp2_036',
 'student_exp2_037',
 'student_exp2_038',
 'student_exp2_039',
 'student_exp2_040',
 'student_exp2_041',
 'student_exp2_042',
 'student_exp2_043',
 'student_exp2_044',
 'student_exp2_045',
 'student_exp2_046',
 'student_exp2_047',
 'student_exp

In [29]:
# create list and then df for the cue words and the participant_ids, repeating them for the n of participants
repeated_cue_words = []
repeated_participant_ids = []
for word in cue_words:
    repeated_cue_words.append(pd.Series([word] * len(association_words)))
    repeated_participant_ids.append(pd.Series(participant_ids))

repeated_cue_words_df = pd.concat(repeated_cue_words, axis=1)
repeated_cue_words_df.columns = range(repeated_cue_words_df.shape[1])

repeated_participant_ids_df = pd.concat(repeated_participant_ids, axis=1)
repeated_participant_ids_df.columns = [f"participant_{i}" for i in range(repeated_participant_ids_df.shape[1])]

### Merge of dfs and get final df

In [30]:
# merge the cue words and the associated words in a single df

# create list with the final cols order
association_cue_words_cols = []
# iterate over the col of the cue words df
for i in range(repeated_cue_words_df.shape[1]):  
    # take each col of the df 
    association_cue_words_cols.append(repeated_cue_words_df.iloc[:, i])
    # for every cue word column get the starting point every 3 association cols
    start = i * 3
    # get all the data, form the starting point every 3 cols for each cue word, and get the 3 cols from that statring point (0:3, 3:6, 6:9, etc.)
    association_cue_words_cols.append(association_words.iloc[:, start:start+3])

# concatenate the df with the data and assign indices
cue_associations_df = pd.concat(association_cue_words_cols, axis=1)
cue_associations_df.columns = range(cue_associations_df.shape[1])
cue_associations_df

,0,1,2,3,4,5,6,7,8,9,...,198,199,200,201,202,203,204,205,206,207
0,tecnologia,Computer,Innovazione,Sviluppo,matematica,Calcolo,Numeri,Base,ingegneria,Programmazine,...,NaN,NaN,programmare,NaN,NaN,NaN,equazione,NaN,NaN,NaN
1,tecnologia,Computer,Informatica,NaN,matematica,Equazioni,Numeri,scuole superiori,ingegneria,Numeri,...,Numeri,Matematica,programmare,Informatica,Google,Scuole superiori,equazione,Matematica,Numeri,Prova
2,tecnologia,computer,dati,rete,matematica,numeri,difficoltà,esercizi,ingegneria,calcoli,...,NaN,NaN,programmare,dati,NaN,NaN,equazione,numeri,NaN,NaN
3,tecnologia,Computer,Internet,NaN,matematica,Numeri,Calcolare,NaN,ingegneria,NaN,...,NaN,NaN,programmare,NaN,NaN,NaN,equazione,NaN,NaN,NaN
4,tecnologia,computer,task,insegnante,matematica,numeri,lettere,equazioni,ingegneria,laurea,...,NaN,NaN,programmare,informatica,codice,NaN,equazione,matematica,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,tecnologia,Computer,Programmazione,Sviluppo,matematica,Difficile,Numeri,Equazioni,ingegneria,Meccanica,...,soluzioni,codici,programmare,codici,computer,dati,equazione,grado,numeri,calcolo
76,tecnologia,Computer,Informatica,Innovazione,matematica,Numeri,Teoremi,Equazioni,ingegneria,Ingegnere,...,NaN,NaN,programmare,NaN,NaN,NaN,equazione,NaN,NaN,NaN
77,tecnologia,computer,progresso,informatica,matematica,numeri,calcolo,equazione,ingegneria,misure,...,NaN,NaN,programmare,NaN,NaN,NaN,equazione,NaN,NaN,NaN
78,tecnologia,computer,tecna,tasti,matematica,numeri,rosso,operazioni,ingegneria,padre,...,NaN,NaN,programmare,NaN,NaN,NaN,equazione,NaN,NaN,NaN


In [31]:
# insert the valence values for all the cue and associations words
col_order =[]
# alternate one col of words and one col of valence scores
for col1, col2 in zip(cue_associations_df.columns, valence_scores.columns):
    col_order.extend([col1, col2])
# concatenate the two dfs (with words and cols) with the alterated col order
valence_association_raw = pd.concat([cue_associations_df, valence_scores], axis=1)[col_order]
valence_association_raw

,0,val_0,1,val_1,2,val_2,3,val_3,4,val_4,...,203,val_203,204,val_204,205,val_205,206,val_206,207,val_207
0,tecnologia,3,Computer,3,Innovazione,4,Sviluppo,4,matematica,4,...,NaN,NaN,equazione,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,tecnologia,NaN,Computer,4,Informatica,3,NaN,NaN,matematica,3,...,Scuole superiori,4,equazione,3,Matematica,3,Numeri,3,Prova,3
2,tecnologia,3,computer,3,dati,3,rete,3,matematica,2,...,NaN,NaN,equazione,NaN,numeri,NaN,NaN,NaN,NaN,NaN
3,tecnologia,3,Computer,3,Internet,2,NaN,NaN,matematica,3,...,NaN,NaN,equazione,3,NaN,NaN,NaN,NaN,NaN,NaN
4,tecnologia,2,computer,3,task,2,insegnante,4,matematica,2,...,NaN,NaN,equazione,2,matematica,2,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,tecnologia,4,Computer,3,Programmazione,2,Sviluppo,5,matematica,2,...,dati,4,equazione,2,grado,2,numeri,2,calcolo,2
76,tecnologia,3,Computer,3,Informatica,3,Innovazione,4,matematica,4,...,NaN,NaN,equazione,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,tecnologia,4,computer,3,progresso,4,informatica,2,matematica,3,...,NaN,NaN,equazione,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,tecnologia,2,computer,3,tecna,5,tasti,2,matematica,2,...,NaN,NaN,equazione,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
# concatenate each col of the participant_ids every 8 cols with the association and valence df
concat_cols = [
    # get each repeated_ids col (for each cue word)
    pd.concat([repeated_participant_ids_df.iloc[:, [i]], 
               # get the cue word, ass words and all valences
               valence_association_raw.iloc[:, i*8:(i+1)*8]], axis=1)
    for i in range(repeated_participant_ids_df.shape[1])
]

# the concatenate all the chunks of data concated by concat_cols into a single df
concatened_df = pd.concat(concat_cols, axis=1)

In [33]:
# reshape the df to get the final one with 9 cols and as rows the n of cue words * n participants
# taking -1 to get all the rows
reshape = concatened_df.to_numpy().reshape(-1, 9)
# get the reshaped numpy array into a df
ass_val_raw_df = pd.DataFrame(reshape)
# assign the names of cols
ass_val_raw_df.columns = ['participant_id', 'cue_word', 'valence_cue_word', 'association_1', 'valence_association_1', 'association_2', 'valence_association_2', 'association_3', 'valence_association_3']

# sort values by cue word and participant_ids
ass_val_raw_df.sort_values(by=['cue_word', 'participant_id']).reset_index(drop=True)

,participant_id,cue_word,valence_cue_word,association_1,valence_association_1,association_2,valence_association_2,association_3,valence_association_3
0,student_exp2_001,algoritmo,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,student_exp2_002,algoritmo,3,Operazioni,3,Numeri,3,Matematica,3
2,student_exp2_003,algoritmo,NaN,dati,NaN,NaN,NaN,NaN,NaN
3,student_exp2_004,algoritmo,3,Istruzione,4,NaN,NaN,NaN,NaN
4,student_exp2_005,algoritmo,3,informatica,3,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
4155,student_exp2_076,utilità_della_matematica,2,parziale,2,approfondire,4,applicazioni,4
4156,student_exp2_077,utilità_della_matematica,3,Numero,3,Mancanza,2,NaN,NaN
4157,student_exp2_078,utilità_della_matematica,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4158,student_exp2_079,utilità_della_matematica,3,rosso,3,NaN,NaN,NaN,NaN


## Associations and valences df cleaning

In [34]:
ass_val_cleaning = ass_val_raw_df.copy()
# get all the associated words in lower case and strip for unwanted spaces
for col in ass_val_raw_df.columns[3::2]:
    ass_val_cleaning[col] = ass_val_raw_df[col].where(
        ass_val_raw_df[col].isna(),  # except if they're NaN values
        ass_val_raw_df[col].astype(str).str.lower().str.strip())

In [35]:
# get the indices of the second rows related to the cue word 'rimandare', which was duplicated
duplicate_indices = ass_val_cleaning[ass_val_cleaning['cue_word'] == 'rimandare'].groupby('participant_id').tail(1).index
# remove those rows, so we just keep the first associations made with the word 'rimandare'
ass_val_cleaning = ass_val_cleaning.drop(duplicate_indices).reset_index(drop=True)

In [36]:
# get the percentage of missing data (NaN values) for each participants
missing_values = (
    ass_val_raw_df
    .set_index("participant_id")
    .isna()
    .mean(axis=1)
    .groupby(level=0)
    .mean() * 100                   
)
# make a list of participants to drop based of the percentage of missing data (more than the 40%)
to_drop = missing_values[missing_values > 40].index
print("Participants to drop:", len(to_drop), list(to_drop))

# create a df where we remove all the participants of the list to_drop
associations_valence_df = ass_val_cleaning[~ass_val_cleaning["participant_id"].isin(to_drop)]

Participants to drop: 14 ['student_exp2_003', 'student_exp2_004', 'student_exp2_009', 'student_exp2_048', 'student_exp2_050', 'student_exp2_053', 'student_exp2_054', 'student_exp2_055', 'student_exp2_056', 'student_exp2_058', 'student_exp2_062', 'student_exp2_067', 'student_exp2_069', 'student_exp2_080']


In [37]:
# create a mask to remove all the rows where all the associations cols (3::2) are empty (with NaNs)
mask = ~associations_valence_df.iloc[:, 3::2].isna().all(axis=1)
# in the df keep only rows with at least one association
associations_valence_cleaned = associations_valence_df.loc[mask]

In [38]:
associations_valence_cleaned.sort_values(by=['cue_word', 'participant_id']).reset_index(drop=True)

,participant_id,cue_word,valence_cue_word,association_1,valence_association_1,association_2,valence_association_2,association_3,valence_association_3
0,student_exp2_002,algoritmo,3,operazioni,3,numeri,3,matematica,3
1,student_exp2_005,algoritmo,3,informatica,3,NaN,NaN,NaN,NaN
2,student_exp2_006,algoritmo,2,calcoli,2,matematica,2,numeri,2
3,student_exp2_007,algoritmo,NaN,pc,NaN,informatica,NaN,NaN,NaN
4,student_exp2_008,algoritmo,4,dell'amore,3,marr,4,computazionale,4
...,...,...,...,...,...,...,...,...,...
3069,student_exp2_074,utilità_della_matematica,5,ingegneria,5,quotidianità,5,psicologia,5
3070,student_exp2_075,utilità_della_matematica,3,avanzamento scientifico,4,NaN,NaN,NaN,NaN
3071,student_exp2_076,utilità_della_matematica,2,parziale,2,approfondire,4,applicazioni,4
3072,student_exp2_077,utilità_della_matematica,3,numero,3,mancanza,2,NaN,NaN


In [43]:
associations_valence_cleaned.to_csv(f'{path}/out/formatting/humans_2023/associations_valence_2023_humans.csv', index=False)

In [44]:
# after removing NaN rows and participants with less then 60% of data see how many participants present each cue word
Counter(associations_valence_cleaned['cue_word'])

Counter({'tecnologia': 66,
         'matematica': 66,
         'biologia': 66,
         'fisica': 66,
         'statistica': 66,
         'classe': 66,
         'esame': 66,
         'successo': 66,
         'genitore': 66,
         'compito': 66,
         'numeri': 66,
         'calcolo': 66,
         'memorizzare': 66,
         'scadenza': 65,
         'intelligenza': 65,
         'studente/ssa': 65,
         'insegnante': 65,
         'cambio_di_piani': 65,
         'studiare': 65,
         'pianificare': 65,
         'rimandare': 65,
         'fissare_un_obiettivo': 65,
         'programma': 65,
         'mappa_concettuale': 65,
         'curiosità': 64,
         'note': 64,
         'impegno': 64,
         'motivazione': 64,
         'ingegneria': 63,
         'revisionare': 63,
         'concentrazione': 63,
         'sfida': 63,
         'distrazione': 62,
         'sforzo': 61,
         'confusione': 61,
         'importanza': 60,
         'problema': 60,
         'ansia': 59,


## MAS-IT

In [45]:
# get the part of the excel where the mas-it data is contanined (from row 30, in cols from 57 to 80)
mas_it_raw = data_2023_humans.iloc[30:, 57:80].reset_index(drop=True)
# assign the cols names
mas_it_raw.columns = [f"mas_it_{n}" for n in range(1, 24)]
# insert the participant_id column
mas_it_raw.insert(0, "participant_id", participant_ids)
# create e new df where we keep only the data from the participant id that are not in the to_drop list (created before based on the missing associations/valence data)
mas_it_cleaned = mas_it_raw[~mas_it_raw["participant_id"].isin(to_drop)].reset_index(drop=True)
# export the data to the csv
mas_it_cleaned.to_csv(f'{path}/out/formatting/humans_2023/mas_it_2023_humans.csv')

## 2024 and 2023 data intersection

In [25]:
associations_valence_2024 = pd.read_csv('associations_valence_2024_humans.csv')

In [30]:
cue_words_2023 = associations_valence_cleaned['cue_word'].unique().tolist()
cue_words_2024 = associations_valence_2024['cue_word'].unique().tolist()

print('common words:', set(cue_words_2023).intersection(set(cue_words_2024)))
print('number of common words:', len(set(cue_words_2023).intersection(set(cue_words_2024))))

common words: {'sfida', 'statistica', 'biologia', 'insegnante', 'esame', 'ansia', 'fisica', 'matematica', 'curiosità', 'equazione'}
number of common words: 10


In [39]:
mix_2023_2024_df = pd.concat(
    [associations_valence_2024, associations_valence_cleaned],
    ignore_index=True
)
len(mix_2023_2024_df['participant_id'].unique())

133